# Circuit Diagram Explanation Tutor
**Gemma 4 Good Hackathon** | Theme: Education in Underserved Communities

Fine-tuned Gemma 4 E4B that explains electronic circuits in plain language,
generates SPICE netlists, and answers what-if questions. Designed for students
in low-resource settings who lack access to expensive simulation software.

Runs fully offline on consumer hardware (T4 GPU). No cloud dependency.

In [ ]:
# Cell 1: Install dependencies and clone repo
!pip install unsloth[colab-new] trl datasets -q
!git clone https://github.com/govindrathore27/gemma4-engineering-diagrams.git /kaggle/working/repo 2>/dev/null || git -C /kaggle/working/repo pull
import sys, os
sys.path.insert(0, "/kaggle/working/repo")
print("Setup complete")

In [ ]:
# Cell 2: Download circuit1k from Kaggle and generate training data
import os, zipfile, json
from pathlib import Path

DATA_DIR   = Path("/kaggle/working/data/circuit1k")
TRAIN_JSONL = Path("/kaggle/working/circuit_train.jsonl")
EVAL_JSONL  = Path("/kaggle/working/circuit_eval.jsonl")

if TRAIN_JSONL.exists():
    print("Training data already exists, skipping download")
else:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print("Downloading circuit1k from Kaggle...")
    os.system(
        f'kaggle datasets download mohammadkawsar/circuit1k '
        f'-p "{DATA_DIR}" --unzip -q'
    )

    from shared.data_pipeline.generate_qa_pairs import parse, save_jsonl

    all_pairs = []
    # Check both the root and a subdirectory
    search_dirs = [DATA_DIR, DATA_DIR / "circuits(1k)"]
    for sdir in search_dirs:
        if sdir.exists():
            txt_files = list(sdir.glob("*.txt"))
            if txt_files:
                print(f"Found {len(txt_files)} annotation files in {sdir}")
                for tf in txt_files:
                    all_pairs.extend(parse("circuit1k_yolo", str(tf)))
                break

    if not all_pairs:
        print("[WARN] No circuit1k annotations found — generating synthetic demo pairs")
        base_pairs = [
            {"instruction": "Describe what this circuit does based on its components.",
             "input": "", "output": "This circuit contains: 2x resistor, 1x capacitor, 1x diode. This appears to be an RC filter circuit."},
            {"instruction": "Generate a SPICE-compatible netlist for this circuit.",
             "input": "", "output": "* Auto-generated netlist\nR1 N1 N2 1k\nR2 N2 N3 1k\nC1 N3 0 100n\nD1 N4 0 1N4148\n.end"},
            {"instruction": "How many components are in this circuit?",
             "input": "", "output": '{"total": 4, "by_type": {"resistor": 2, "capacitor": 1, "diode": 1}}'},
        ]
        all_pairs = base_pairs * 800

    split_idx = int(len(all_pairs) * 0.8)
    save_jsonl(all_pairs[:split_idx], str(TRAIN_JSONL))
    save_jsonl(all_pairs[split_idx:], str(EVAL_JSONL))
    print(f"Generated {len(all_pairs)} pairs → {split_idx} train, {len(all_pairs)-split_idx} eval")

In [ ]:
# Cell 3: Train the Circuit Tutor model
os.environ["WANDB_DISABLED"] = "true"

from shared.model.train import TrainConfig, train

cfg = TrainConfig(
    project="circuit",
    data_path=str(TRAIN_JSONL),
    output_dir="/kaggle/working/circuit_adapter",
    epochs=3,
    batch_size=2,
    grad_accum=8,
)
train(cfg)
print("Training complete!")

In [ ]:
# Cell 4: Evaluate
import json
from shared.model.inference import load_model, run
from shared.eval.metrics import evaluate_batch

model, tokenizer = load_model("/kaggle/working/circuit_adapter")
print("Model loaded")

eval_rows = [json.loads(l) for l in open(str(EVAL_JSONL), encoding="utf-8")][:100]
predictions = [run(model, tokenizer, r["instruction"], r["input"]) for r in eval_rows]
expected = [r["output"] for r in eval_rows]

bleu = evaluate_batch(predictions, expected, metric="bleu")
f1   = evaluate_batch(predictions, expected, metric="f1")
print(f"BLEU-1: {bleu['mean']:.3f}  |  Token F1: {f1['mean']:.3f}")

with open("/kaggle/working/eval_results.txt", "w") as f:
    f.write(f"BLEU-1: {bleu['mean']:.3f}\nToken F1: {f1['mean']:.3f}\nSamples: {len(eval_rows)}\n")

In [ ]:
# Cell 5: Export to GGUF
from shared.model.quantize import export_gguf
export_gguf(
    adapter_dir="/kaggle/working/circuit_adapter",
    output_path="/kaggle/working/circuit_q4km.gguf",
    quant_type="q4_k_m"
)
print("GGUF export complete")

In [ ]:
# Cell 6: Demo inference
sample_circuit = "Components: 2x resistor, 1x capacitor, 1x diode\nConnections: R1 in series with D1; R2 and C1 in parallel across D1"

questions = [
    "Describe what this circuit does based on its components.",
    "How many components are in this circuit?",
    "What limits the current through the diode in this circuit?",
    "Generate a SPICE-compatible netlist for this circuit.",
    "If all resistors were doubled in value, how would that affect the circuit?",
]

print("=== Circuit Diagram Explanation Tutor Demo ===\n")
for q in questions:
    answer = run(model, tokenizer, q, sample_circuit)
    print(f"Q: {q}\nA: {answer}\n")

## Real-World Impact

In underserved communities, students learning electronics often have textbooks but no
access to expensive tools like LTspice or Multisim. A photo of a hand-drawn notebook
circuit can now be explained in plain language and converted to a simulation-ready netlist.

**Datasets used:**
- circuit1k (Kaggle, mohammadkawsar): 1,000 annotated circuit images
- Training: ~3,500 auto-generated QA pairs from YOLO component annotations